# Causal Inference

## Learning Objectives
1. Demonstrate Simpson's Paradox and explain why aggregate statistics mislead without stratification
2. Implement propensity score matching using logistic regression and compute the Average Treatment Effect on Treated (ATT)
3. Implement the Difference-in-Differences (DiD) estimator and interpret the interaction coefficient
4. Compare naive, propensity-matched, and regression-adjusted causal estimates to quantify confounding bias

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print("numpy:", np.__version__)
print("sklearn:", __import__('sklearn').__version__)
print("Setup complete.")

## Level 1: Simpson's Paradox

Simpson's Paradox occurs when a trend in aggregate data reverses when data is stratified by a third variable.

This is the simplest and most visceral demonstration that correlation != causation.
The aggregate is misleading because a confounding variable is driving both group membership and the outcome.

In [ ]:
# -----------------------------------------------------------------------
# Level 1: Simpson's Paradox -- aggregate trend reverses in subgroups
# -----------------------------------------------------------------------

def generate_simpsons_data(n_mild: int = 700, n_severe: int = 300,
                           seed: int = 42) -> dict:
    """
    Generate Simpson's Paradox dataset: treatment A vs B, with disease severity as confounder.

    Ground truth: Treatment B is better within each severity group.
    Aggregate illusion: Treatment A appears better overall because it is disproportionately
    assigned to mild cases (lower severity -> better outcomes regardless of treatment).

    Returns
    -------
    dict with arrays for treatment, severity, survival, and group-level statistics
    """
    rng = np.random.default_rng(seed)

    # Mild cases: 70% of sample
    # Treatment B is better: survival 90% vs A's 85%
    # Most mild cases got Treatment A (systematic bias due to historical practice)
    n_mild_A = int(n_mild * 0.80)   # 80% of mild cases got A
    n_mild_B = n_mild - n_mild_A

    survival_mild_A = rng.binomial(1, 0.85, n_mild_A)    # 85% survival, A, mild
    survival_mild_B = rng.binomial(1, 0.90, n_mild_B)    # 90% survival, B, mild (B is better!)

    # Severe cases: 30% of sample
    # Treatment B is still better: 60% vs A's 50%
    # Most severe cases got Treatment B (preferentially assigned to harder cases)
    n_severe_A = int(n_severe * 0.20)  # only 20% of severe got A
    n_severe_B = n_severe - n_severe_A

    survival_severe_A = rng.binomial(1, 0.50, n_severe_A)  # 50% survival, A, severe
    survival_severe_B = rng.binomial(1, 0.60, n_severe_B)  # 60% survival, B, severe (B better!)

    # Combine
    survival = np.concatenate([survival_mild_A, survival_mild_B,
                                survival_severe_A, survival_severe_B])
    treatment = np.concatenate([np.zeros(n_mild_A), np.ones(n_mild_B),
                                 np.zeros(n_severe_A), np.ones(n_severe_B)])
    severity = np.concatenate([np.zeros(n_mild_A + n_mild_B),
                                np.ones(n_severe_A + n_severe_B)])

    # Compute rates
    mask_A, mask_B = treatment == 0, treatment == 1
    overall_A = survival[mask_A].mean()
    overall_B = survival[mask_B].mean()

    mild_A  = survival_mild_A.mean()
    mild_B  = survival_mild_B.mean()
    sev_A   = survival_severe_A.mean()
    sev_B   = survival_severe_B.mean()

    return {
        "survival": survival, "treatment": treatment, "severity": severity,
        "overall_A": overall_A, "overall_B": overall_B,
        "mild_A": mild_A, "mild_B": mild_B,
        "severe_A": sev_A, "severe_B": sev_B,
        "n_mild_A": n_mild_A, "n_mild_B": n_mild_B,
        "n_severe_A": n_severe_A, "n_severe_B": n_severe_B,
    }


data = generate_simpsons_data()

print("Simpson's Paradox: Treatment A vs B, outcome = survival")
print()
print("AGGREGATE (ignoring severity):")
print(f"  Treatment A survival: {data['overall_A']:.3f}  <-- appears BETTER")
print(f"  Treatment B survival: {data['overall_B']:.3f}  <-- appears WORSE")
print(f"  Apparent conclusion: USE TREATMENT A")
print()
print("STRATIFIED BY SEVERITY (the truth):")
print(f"  Mild cases:   A={data['mild_A']:.3f}  B={data['mild_B']:.3f}  (B is better by {data['mild_B']-data['mild_A']:+.3f})")
print(f"  Severe cases: A={data['severe_A']:.3f}  B={data['severe_B']:.3f}  (B is better by {data['severe_B']-data['severe_A']:+.3f})")
print(f"  True conclusion: USE TREATMENT B")
print()
print("WHY THE PARADOX:")
print(f"  Mild cases ({data['n_mild_A']+data['n_mild_B']} total): {data['n_mild_A']} got A, {data['n_mild_B']} got B")
print(f"  Severe cases ({data['n_severe_A']+data['n_severe_B']} total): {data['n_severe_A']} got A, {data['n_severe_B']} got B")
print("  Severity drives BOTH treatment assignment AND outcome -- it is a confounder")
print("  Ignoring it conflates the treatment effect with the severity effect")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
categories = ["Mild (A)", "Mild (B)", "Severe (A)", "Severe (B)", "Overall (A)", "Overall (B)"]
values = [data['mild_A'], data['mild_B'], data['severe_A'], data['severe_B'],
          data['overall_A'], data['overall_B']]
colors_bar = ["steelblue", "darkorange", "steelblue", "darkorange", "steelblue", "darkorange"]
bars = axes[0].bar(categories, values, color=colors_bar, alpha=0.8, edgecolor="white")
axes[0].set_ylabel("Survival Rate", fontsize=12)
axes[0].set_title("Simpson's Paradox: Survival by Group", fontsize=12, fontweight="bold")
axes[0].set_ylim(0, 1.1)
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", fontsize=9)
from matplotlib.patches import Patch
legend_elems = [Patch(facecolor="steelblue", label="Treatment A"),
                Patch(facecolor="darkorange", label="Treatment B")]
axes[0].legend(handles=legend_elems, fontsize=9)
axes[0].grid(alpha=0.3, axis="y")
axes[0].annotate("B better\nin both\nsubgroups!", xy=(3, 0.62), fontsize=10, color="green",
                 fontweight="bold")
axes[0].annotate("A 'better'\naggregate!", xy=(4, 0.84), fontsize=10, color="red",
                 fontweight="bold")

axes[1].set_visible(False)  # keep second panel blank for clarity

plt.tight_layout()
plt.savefig("simpsons_paradox.png", dpi=100, bbox_inches="tight")
plt.show()

## Level 2: Propensity Score Matching

Propensity scores P(treated | X) estimated via logistic regression.
We match each treated unit to the closest control by propensity score.
Goal: create a matched control group that "looks like" the treated group on observables.

In [ ]:
# -----------------------------------------------------------------------
# Level 2: Propensity score matching (PSM)
# -----------------------------------------------------------------------

def generate_observational_data(n: int = 1000, seed: int = 42) -> dict:
    """
    Generate observational data with confounded treatment assignment.

    Setting: job training program. Treated = attended training.
    Confounder: prior_income (low income -> more likely to attend AND lower post-income baseline)
    True causal effect of training: +$2000

    Without adjustment, naive estimate will be biased downward (treated group
    has lower baseline income, pulling their outcomes down).
    """
    rng = np.random.default_rng(seed)
    n_treated = n // 3  # only 1/3 attend training (selection bias: poorer people attend more)

    # Prior income: treated group systematically lower (confounding)
    prior_income_treated = rng.normal(30000, 8000, n_treated)
    prior_income_control = rng.normal(40000, 8000, n - n_treated)

    # Age as additional covariate (younger people slightly more likely to train)
    age_treated = rng.normal(28, 5, n_treated)
    age_control = rng.normal(33, 6, n - n_treated)

    # Post-training income: depends on prior income + treatment effect
    true_effect = 2000
    noise = 5000
    income_treated = prior_income_treated * 1.05 + true_effect + rng.normal(0, noise, n_treated)
    income_control = prior_income_control * 1.05                + rng.normal(0, noise, n - n_treated)

    return {
        "prior_income": np.concatenate([prior_income_treated, prior_income_control]),
        "age":          np.concatenate([age_treated, age_control]),
        "post_income":  np.concatenate([income_treated, income_control]),
        "treated":      np.concatenate([np.ones(n_treated), np.zeros(n - n_treated)]),
        "true_ate":     true_effect,
        "n_treated":    n_treated,
    }


def propensity_score_matching(data: dict) -> dict:
    """
    Estimate propensity scores via logistic regression, then match treated to controls.

    1. Fit logistic regression: P(treated | prior_income, age)
    2. Match each treated unit to the control with closest propensity score
    3. Compute ATT = mean(Y_treated - Y_matched_control)
    """
    treated = data["treated"].astype(int)
    X = np.column_stack([data["prior_income"], data["age"]])
    y = data["post_income"]

    # Estimate propensity scores
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X_scaled, treated)
    pscore = lr.predict_proba(X_scaled)[:, 1]

    # Naive estimate (ignores confounding)
    naive_ate = y[treated == 1].mean() - y[treated == 0].mean()

    # Propensity score matching (nearest neighbor, 1:1, with replacement)
    treated_idx = np.where(treated == 1)[0]
    control_idx = np.where(treated == 0)[0]
    pscore_treated = pscore[treated_idx]
    pscore_control = pscore[control_idx]

    matched_outcomes = []
    for ps_t, idx_t in zip(pscore_treated, treated_idx):
        # Find closest control by propensity score
        distances = np.abs(pscore_control - ps_t)
        best_match = control_idx[np.argmin(distances)]
        matched_outcomes.append(y[best_match])

    att = y[treated_idx].mean() - np.mean(matched_outcomes)

    # Covariate balance before/after matching
    smd_before_income = ((X[treated == 1, 0].mean() - X[treated == 0, 0].mean()) /
                          np.sqrt((X[treated == 1, 0].var() + X[treated == 0, 0].var()) / 2))
    matched_x_income = X[control_idx[np.argmin(np.abs(pscore_control - ps_t))], 0]
    # Simplified: just report before/after
    smd_before_age = ((X[treated == 1, 1].mean() - X[treated == 0, 1].mean()) /
                       np.sqrt((X[treated == 1, 1].var() + X[treated == 0, 1].var()) / 2))

    return {
        "naive_ate": naive_ate,
        "att_psm": att,
        "true_ate": data["true_ate"],
        "pscore": pscore,
        "smd_income_before": smd_before_income,
        "smd_age_before": smd_before_age,
        "bias_naive": naive_ate - data["true_ate"],
        "bias_psm": att - data["true_ate"],
    }


obs_data = generate_observational_data(n=1000)
result_psm = propensity_score_matching(obs_data)

print(f"Propensity Score Matching: Training Effect on Income")
print(f"  True causal effect (ATE):  ${result_psm['true_ate']:,.0f}")
print(f"  Naive estimate:            ${result_psm['naive_ate']:,.0f}  (bias: ${result_psm['bias_naive']:+,.0f})")
print(f"  PSM estimate (ATT):        ${result_psm['att_psm']:,.0f}  (bias: ${result_psm['bias_psm']:+,.0f})")
print()
print("Covariate imbalance before matching (SMD, goal < 0.10):")
print(f"  Prior income SMD: {result_psm['smd_income_before']:.3f}  {'IMBALANCED' if abs(result_psm['smd_income_before']) > 0.1 else 'balanced'}")
print(f"  Age SMD:          {result_psm['smd_age_before']:.3f}  {'IMBALANCED' if abs(result_psm['smd_age_before']) > 0.1 else 'balanced'}")
print("PSM reduces these imbalances by finding comparable controls")

## Real-World Example 1: Difference-in-Differences (DiD)

DiD exploits a natural experiment: one group is exposed to a policy/treatment at time T,
another is not. Under the parallel trends assumption, the DiD estimator is:

DiD = (Y_treated_post - Y_treated_pre) - (Y_control_post - Y_control_pre)

This is equivalent to the interaction coefficient in: Y = beta0 + beta1*Treated + beta2*Post + beta3*(Treated*Post) + error

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 1: Difference-in-Differences estimator
# -----------------------------------------------------------------------

def generate_did_data(n_cities: int = 50, n_time: int = 24,
                      treatment_time: int = 12, true_effect: float = 5.0,
                      seed: int = 42) -> dict:
    """
    Generate panel data for DiD analysis.

    Setting: minimum wage policy change in treated cities (first 25).
    Outcome: monthly employment index.
    Control cities: no policy change.
    True causal effect: +5 employment index units after policy.

    Parallel trends: both groups have same time trend before treatment.
    """
    rng = np.random.default_rng(seed)
    n_treated_cities = n_cities // 2

    records = []
    for city in range(n_cities):
        treated = int(city < n_treated_cities)
        baseline = rng.normal(100, 10)  # city-specific baseline level
        trend = 0.3  # common time trend (satisfies parallel trends assumption)
        noise_sd = 2.0

        for t in range(n_time):
            post = int(t >= treatment_time)
            y = (baseline + trend * t +
                 true_effect * treated * post +   # causal effect
                 rng.normal(0, noise_sd))
            records.append({"city": city, "t": t, "treated": treated,
                             "post": post, "y": y})

    return records


def estimate_did(records: list) -> dict:
    """
    Estimate DiD via: (1) manual 2x2 table, (2) OLS regression with interaction.
    """
    import numpy as np

    # Group records
    def avg_y(group_records):
        return np.mean([r["y"] for r in group_records])

    treated_pre  = [r for r in records if r["treated"] == 1 and r["post"] == 0]
    treated_post = [r for r in records if r["treated"] == 1 and r["post"] == 1]
    control_pre  = [r for r in records if r["treated"] == 0 and r["post"] == 0]
    control_post = [r for r in records if r["treated"] == 0 and r["post"] == 1]

    y_tp = avg_y(treated_post)
    y_tb = avg_y(treated_pre)
    y_cp = avg_y(control_post)
    y_cb = avg_y(control_pre)

    did_manual = (y_tp - y_tb) - (y_cp - y_cb)

    # OLS regression: Y = b0 + b1*Treated + b2*Post + b3*(Treated*Post)
    y = np.array([r["y"] for r in records])
    treated_arr = np.array([r["treated"] for r in records])
    post_arr = np.array([r["post"] for r in records])
    interaction = treated_arr * post_arr
    X_did = np.column_stack([np.ones(len(y)), treated_arr, post_arr, interaction])

    beta, _, _, _ = np.linalg.lstsq(X_did, y, rcond=None)

    return {
        "did_manual": did_manual,
        "did_ols": beta[3],       # interaction coefficient = DiD estimate
        "beta_treated": beta[1],  # time-invariant treated group difference
        "beta_post": beta[2],     # common time trend effect
        "y_tp": y_tp, "y_tb": y_tb, "y_cp": y_cp, "y_cb": y_cb,
    }


records = generate_did_data(n_cities=50, n_time=24, treatment_time=12, true_effect=5.0)
did_result = estimate_did(records)

print("Difference-in-Differences: Minimum Wage Policy Effect")
print()
print("2x2 Table (mean employment index):")
print(f"{'':>20}  {'Pre-Policy':>12}  {'Post-Policy':>12}  {'Difference':>12}")
print("-" * 60)
print(f"{'Treated cities':<20}  {did_result['y_tb']:>12.2f}  {did_result['y_tp']:>12.2f}  {did_result['y_tp']-did_result['y_tb']:>+11.2f}")
print(f"{'Control cities':<20}  {did_result['y_cb']:>12.2f}  {did_result['y_cp']:>12.2f}  {did_result['y_cp']-did_result['y_cb']:>+11.2f}")
print(f"{'DiD estimate':<20}  {'':>12}  {'':>12}  {did_result['did_manual']:>+11.2f}")
print()
print(f"True causal effect: +5.00")
print(f"Manual DiD:         {did_result['did_manual']:+.3f}")
print(f"OLS interaction:    {did_result['did_ols']:+.3f}")
print()
print("OLS regression coefficients:")
print(f"  beta_treated (group level diff): {did_result['beta_treated']:.3f}")
print(f"  beta_post (common time trend):   {did_result['beta_post']:.3f}")
print(f"  beta_interaction (DiD = causal): {did_result['did_ols']:.3f}")

## Real-World Example 2: Confounding Control Comparison

Three approaches to estimate the training effect:
1. **Naive**: simply compare treated vs control (biased by confounding)
2. **Regression adjustment**: control for confounders in OLS (assumes linearity)
3. **PSM**: match treated to comparable controls (non-parametric)

We compare all three to the true effect.

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 2: Naive vs regression adjustment vs PSM comparison
# -----------------------------------------------------------------------

def regression_adjustment(data: dict) -> float:
    """
    OLS regression controlling for confounders.
    Y = beta0 + beta1*Treated + beta2*PriorIncome + beta3*Age + error
    beta1 is the regression-adjusted treatment effect.
    """
    y = data["post_income"]
    treated = data["treated"]
    prior_income = data["prior_income"]
    age = data["age"]

    X_ols = np.column_stack([np.ones(len(y)), treated, prior_income, age])
    beta, _, _, _ = np.linalg.lstsq(X_ols, y, rcond=None)
    return beta[1]  # coefficient on Treated


# Run all three methods
obs_data_2 = generate_observational_data(n=2000, seed=99)
naive_est = obs_data_2["post_income"][obs_data_2["treated"] == 1].mean() -             obs_data_2["post_income"][obs_data_2["treated"] == 0].mean()
reg_adj_est = regression_adjustment(obs_data_2)
psm_result_2 = propensity_score_matching(obs_data_2)
true_effect = obs_data_2["true_ate"]

print("Comparison: Estimating Training Effect on Income (n=2000)")
print(f"True causal effect: ${true_effect:,.0f}")
print()
print(f"{'Method':<25}  {'Estimate':>10}  {'Bias':>10}  {'% Bias':>10}")
print("-" * 55)
methods_est = [
    ("Naive comparison",      naive_est),
    ("Regression adjustment", reg_adj_est),
    ("Propensity Score Match", psm_result_2["att_psm"]),
]
for name, est in methods_est:
    bias = est - true_effect
    pct_bias = bias / true_effect * 100
    flag = "OK" if abs(pct_bias) < 20 else "BIASED"
    print(f"{name:<25}  ${est:>9,.0f}  ${bias:>9,+.0f}  {pct_bias:>9.1f}%  {flag}")

print()
print("Key insights:")
print("  Naive: Largest bias -- ignores that treated group has lower prior income")
print("  Regression: Good if linearity holds and all confounders are measured")
print("  PSM: Good if overlap assumption holds and all confounders are measured")
print("  All methods fail if key confounders are unmeasured (unobserved confounding)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
estimates = [naive_est, reg_adj_est, psm_result_2["att_psm"]]
method_names = ["Naive", "Regression Adjustment", "Propensity Score Match"]
colors_est = ["crimson", "steelblue", "seagreen"]

ax1 = axes[0]
bars_est = ax1.bar(method_names, estimates, color=colors_est, alpha=0.8, edgecolor="white")
ax1.axhline(true_effect, color="black", linewidth=2.5, linestyle="--", label=f"True effect: ${true_effect:,}")
for bar, est in zip(bars_est, estimates):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f"${est:,.0f}", ha="center", fontsize=10, fontweight="bold")
ax1.set_ylabel("Estimated Training Effect ($)", fontsize=12)
ax1.set_title("Causal Methods vs True Effect", fontsize=13, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3, axis="y")
ax1.set_ylim(min(0, min(estimates) - 500), max(estimates) + 1000)

# Propensity score distribution
ax2 = axes[1]
pscore_all = psm_result_2  # reuse earlier computation
# Recompute pscore for visualization
treated_arr = obs_data_2["treated"].astype(int)
X_v = np.column_stack([obs_data_2["prior_income"], obs_data_2["age"]])
scaler_v = StandardScaler()
X_vs = scaler_v.fit_transform(X_v)
lr_v = LogisticRegression(random_state=42, max_iter=1000)
lr_v.fit(X_vs, treated_arr)
pscore_v = lr_v.predict_proba(X_vs)[:, 1]

ax2.hist(pscore_v[treated_arr == 0], bins=30, alpha=0.6, color="steelblue",
         label="Control", density=True)
ax2.hist(pscore_v[treated_arr == 1], bins=30, alpha=0.6, color="darkorange",
         label="Treated", density=True)
ax2.set_xlabel("Propensity Score", fontsize=12)
ax2.set_ylabel("Density", fontsize=12)
ax2.set_title("Propensity Score Distribution\n(overlap needed for matching)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("causal_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

## Real-World Example 3 + Comparison: DAG Nodes and Collider Bias

A DAG (Directed Acyclic Graph) encodes causal structure.
Controlling for different variable types has fundamentally different effects:
- **Confounder**: control removes bias
- **Mediator**: control blocks causal path (removes part of effect you want)
- **Collider**: control INTRODUCES bias (opens spurious path)

In [ ]:
# -----------------------------------------------------------------------
# Real-World Example 3 + Comparison: Collider bias simulation
#   and summary comparison of all methods
# -----------------------------------------------------------------------

def simulate_collider_bias(n: int = 5000, seed: int = 42) -> dict:
    """
    Simulate collider bias.

    DAG: Exercise (X) -> Health (Y)   [true causal effect]
          X -> Enrolled in gym (C)    [X causes C]
          Y -> Enrolled in gym (C)    [Y causes C]  -- C is a COLLIDER

    True: Exercise has a POSITIVE effect on health.
    Naive (unadjusted): positive relationship visible.
    Incorrect (controlling for collider C): introduces spurious NEGATIVE
      association within gym-enrolled subgroup.
    """
    rng = np.random.default_rng(seed)
    exercise = rng.normal(5, 2, n)          # hours/week
    true_health_effect = 2.0
    health = true_health_effect * exercise + rng.normal(0, 5, n)

    # Collider: probability of gym enrollment increases with both exercise AND health
    logit_gym = -4 + 0.4 * exercise + 0.1 * health
    prob_gym = 1 / (1 + np.exp(-logit_gym))
    gym_enrolled = rng.binomial(1, prob_gym).astype(bool)

    # True effect estimate (no conditioning on collider)
    slope_full, _, _, _, _ = stats.linregress(exercise, health)

    # Biased estimate (conditioning on gym enrolled only)
    slope_cond, _, _, _, _ = stats.linregress(exercise[gym_enrolled], health[gym_enrolled])

    return {
        "exercise": exercise,
        "health": health,
        "gym_enrolled": gym_enrolled,
        "true_effect": true_health_effect,
        "slope_full": slope_full,
        "slope_gym": slope_cond,
        "bias_from_collider": slope_cond - true_health_effect
    }


collider_result = simulate_collider_bias()

print("Collider Bias: Exercise -> Health, Gym enrollment is a collider")
print(f"  True effect of exercise on health: {collider_result['true_effect']:.2f}")
print(f"  Estimated (full sample):           {collider_result['slope_full']:.3f}  <- close to truth")
print(f"  Estimated (gym members only):      {collider_result['slope_gym']:.3f}  <- collider bias!")
print(f"  Bias introduced by conditioning:   {collider_result['bias_from_collider']:+.3f}")
print()
print("WHY: Among gym members, high exercise has fewer extremely unhealthy people")
print("(unhealthy people without exercise wouldn't join a gym)")
print("This artificial restriction creates a spurious negative correlation")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ex = collider_result["exercise"]
he = collider_result["health"]
gym = collider_result["gym_enrolled"]

ax1 = axes[0]
ax1.scatter(ex[~gym], he[~gym], alpha=0.15, s=10, color="steelblue", label="Non-gym")
ax1.scatter(ex[gym],  he[gym],  alpha=0.25, s=10, color="darkorange", label="Gym enrolled")
x_line = np.linspace(ex.min(), ex.max(), 100)
ax1.plot(x_line, collider_result['slope_full'] * x_line + (he.mean() - collider_result['slope_full'] * ex.mean()),
         "k-", linewidth=2.5, label=f"Full sample (slope={collider_result['slope_full']:.2f})")
ax1.plot(x_line[50:], collider_result['slope_gym'] * x_line[50:] + (he[gym].mean() - collider_result['slope_gym'] * ex[gym].mean()),
         "r--", linewidth=2.5, label=f"Gym only (slope={collider_result['slope_gym']:.2f})")
ax1.set_xlabel("Exercise (hrs/week)", fontsize=12)
ax1.set_ylabel("Health Score", fontsize=12)
ax1.set_title("Collider Bias: Exercise vs Health", fontsize=12, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Summary comparison bar chart: all causal methods
ax2 = axes[1]
summary_methods = ["RCT (gold std)", "DiD (natural exp)", "PSM (observed conf)", "Reg Adj (linear)", "Naive (no adj)"]
summary_bias = [0.0, abs(did_result['did_ols'] - 5.0), abs(psm_result_2['bias_psm']),
                abs(reg_adj_est - true_effect), abs(naive_est - true_effect)]
summary_colors = ["gold", "seagreen", "steelblue", "mediumpurple", "crimson"]

bars_s = ax2.bar(summary_methods, summary_bias, color=summary_colors, alpha=0.85, edgecolor="white")
for bar, bias_val in zip(bars_s, summary_bias):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f"${bias_val:,.0f}", ha="center", fontsize=9, fontweight="bold")
ax2.set_ylabel("Absolute Bias ($)", fontsize=12)
ax2.set_title("Causal Method Bias Comparison\n(lower is better)", fontsize=12, fontweight="bold")
ax2.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("causal_full_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

print("\nFinal summary: Causal method selection guide")
print("  RCT: zero bias by design -- use when randomization is possible")
print("  DiD: low bias if parallel trends holds -- use for policy changes")
print("  PSM: low bias if all confounders observed -- use for observational data")
print("  Regression: good if linearity holds -- fast and interpretable")
print("  Naive: always biased in observational settings -- never use for causal claims")